## Homework 1: Agentic RAG

### In this homework, we build a RAG system from scratch and then make it agentic - the same path as the module.
### Instead of the course FAQ, our knowledge base is the course lessons themselves.

First, we will pull the lesson pages straight from the course repository. We will use the commit 8c1834d to make sure everyone works with the exact same data.

In [1]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

GithubRepositoryDataReader downloads the entire repository and goes over all the files in it. Because we specify allowed_extensions={"md"}, it only checks the markdown files.

We also pass a filename_filter so we don't grab every markdown file in the repo, like the top-level README. The lesson pages all live under a module's lessons/ folder, so filtering on /lessons/ keeps just those.

Each file has a parse() method that returns a dictionary with its filename and content:

In [3]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

## Q1. How many lesson pages


In [5]:
len(documents)

72

In [6]:
documents[0]

{'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a simp

## Answer: 72

## Q2. Indexing and searching


Index the documents with minsearch - make content a text field and filename a keyword field. Then search with this query:
```
How does the agentic loop keep calling the model until it stops?
```

In [7]:
from minsearch import Index

index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

index.fit(documents)

In [9]:
question = "How does the agentic loop keep calling the model until it stops?"

search_results = index.search(
    question,
    # boost_dict={"content": 2.0, "section": 0.5},
    # filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

search_results[0]['filename']

'01-agentic-rag/lessons/14-agentic-loop.md'

## Q2. What's the filename of the first result?

## Answer: '01-agentic-rag/lessons/14-agentic-loop.md'

## Q3. RAG

In [11]:
from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
openai_client = OpenAI()

In [16]:
from rag_helper import RAGBase
from openai import OpenAI

class RAGCourse(RAGBase):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
    
    def search(self, query, num_results=5):

        return self.index.search(
            query,
            num_results=num_results,
        )

    def build_context(self, search_results):
        lines = []

        for doc in search_results:
            lines.append(doc["content"])
            lines.append("")

        return "\n".join(lines).strip()

    def llm(self, prompt):
        input_messages = [
            {"role": "developer", "content": self.instructions},
            {"role": "user", "content": prompt}
        ]

        response = self.llm_client.responses.create(
            model=self.model,
            input=input_messages
        )

        return response.output_text, response.usage

openai_client = OpenAI()

rag_course = RAGCourse(index=index, llm_client=openai_client, model='gpt-5.4-mini')

In [17]:
# How does the agentic loop keep calling the model until it stops?
query = "How does the agentic loop keep calling the model until it stops?"

answer = rag_course.rag(query)
answer

('It keeps calling the model in a `while True` loop, and after each call it checks whether the response included any `function_call` items.\n\n- If there **is** a function call, the code runs the tool, appends the tool result to `messages`, and loops again.\n- If there are **no function calls**, it `break`s out of the loop.\n\nSo the stop condition is: **the model returns a final message with no more tool calls**.',
 ResponseUsage(input_tokens=7036, input_tokens_details=InputTokensDetails(cached_tokens=6912), output_tokens=101, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=7137))

## Q3. Use gpt-5.4-mini. How many input (prompt) tokens did we send to the model for this request?
## Answer: 7000


## Q4. Chunking

The lesson pages are long - some are thousands of characters. Long documents make retrieval less precise: a match deep inside a page still pulls in the whole page. A common fix is chunking: split each page into smaller, overlapping pieces and index those instead.

gitsource has a helper for this: chunk_documents. It uses a sliding window - a window of size characters slides across the text in steps of step characters, and each window position becomes one chunk:

In [18]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

With size=2000 and step=1000 (you can see the implementation here):

Each chunk is a window of size characters of the page.
- The window moves forward by step characters between chunks. Since step is smaller than size, consecutive chunks overlap by size - step (1000) characters, so a passage split across a boundary still appears whole in one of the chunks.
- Every chunk keeps the original fields (filename) and adds start (the offset in the page) and content (the chunk text).


In [19]:
len(chunks)

295

## Q4. How many chunks do you get?
## Answer: 295

In [23]:
chunks[50]

{'start': 1000,
 'content': 'decides when to call it and what to search for.\n\nThe same typo question now goes like this:\n\n```mermaid\nflowchart TD\n    U([User: How do I run Olama?])\n    L1[LLM: I\'ll search for \'Olama\']\n    S1[search - Olama - no useful results]\n    L2[LLM: Hmm, no results. Maybe a typo for \'Ollama\'?]\n    S2[search - Ollama - found results!]\n    A([LLM: Here\'s how to run Ollama locally...])\n\n    U --> L1 --> S1 --> L2 --> S2 --> A\n```\n\nThe LLM searched, saw the results were bad, and decided to try again\nwith a different query. It made that decision on its own. We didn\'t\nwrite any code to handle typos.\n\nThe difference is about who makes the decisions:\n\n- With RAG, the developer decides. We fix the steps up front, so\n  search always runs once with the exact user query.\n- With an agent, the LLM decides. It chooses which actions to take\n  and when to stop.\n\nThe mechanism that makes this possible is function calling, and that\'s\nwhat the res

## Q5: RAG with chunking

Chunking makes each request smaller, because we send a smaller context to the LLM. Let's measure that.

Index the chunks from Q4 (same as before: content as a text field, filename as a keyword field), point your RAG at the chunk index, and answer the same query again - reading the input tokens the same way as in Q3.

In [24]:
index_chunking = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

index_chunking.fit(chunks)

openai_client = OpenAI()

rag_course_chunking = RAGCourse(index=index_chunking, llm_client=openai_client, model='gpt-5.4-mini')

In [25]:
# How does the agentic loop keep calling the model until it stops?
query = "How does the agentic loop keep calling the model until it stops?"

answer_for_chunking = rag_course_chunking.rag(query)
answer_for_chunking

('It keeps looping with a `while True` and checks whether the model made any `function_call`s on that turn.\n\n- If there is at least one `function_call`, the code runs the tool, appends the result to `messages`, and continues.\n- If there are no `function_call`s, `has_function_calls` stays `False`, and the loop breaks.\n\nSo the stop condition is: **no function calls in the model’s response**.',
 ResponseUsage(input_tokens=2221, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=97, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=2318))

## Q5: Compare the input tokens with Q3. How many fewer input tokens does the chunked version send?
## Answer: x3 times fewer

## Q6. Turning it into an agent

In [27]:
import json

In [28]:
def search(query):

    return index.search(
        query,
        num_results=5,
    )

In [26]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the lessons database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course lessons."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [29]:
def make_call(call):
    args = json.loads(call.arguments)

    if call.name == "search":
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json,
    }

In [30]:
INSTRUCTIONS = """You're a course teaching assistant. Answer the student's question using the search tool. Make multiple searches with different keywords before answering."""

In [31]:
def agent_loop(instructions, question, model="gpt-5.4-mini") -> str:
    messages = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": question}
    ]

    it = 1

    while True:
        print(f"iteration #{it}...")
        has_function_calls = False

        response = openai_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool]
        )

        messages.extend(response.output)

        for item in response.output:
            if item.type == "function_call":
                print("function_call:", item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == "message":
                print("ASSISTANT:")
                last_answer = item.content[0].text
                print(item.content[0].text)

        it = it + 1
        if has_function_calls == False:
            break

    return last_answer

In [33]:
query = "How does the agentic loop work, and how is it different from plain RAG?"
agent_loop(INSTRUCTIONS, query, model="gpt-5.4-mini")

iteration #1...
function_call: search {"query":"agentic loop RAG difference"}
function_call: search {"query":"agentic loop"}
function_call: search {"query":"plain RAG"}
function_call: search {"query":"RAG agentic"}
iteration #2...
ASSISTANT:
The **agentic loop** is the repeated cycle where the LLM is allowed to decide what to do next:

1. you send the user question plus history to the model,
2. the model may return a **function call** instead of a final answer,
3. your code runs that tool,
4. you send the tool result back to the model,
5. repeat until the model returns an answer with **no more tool calls**.

So the LLM is in the driver’s seat. The loop keeps going until it decides it has enough information. The course calls this the core of an agent: **instructions + tools + memory**, all wired together in a `while` loop.

### How it differs from plain RAG

**Plain RAG** is fixed:

- search once,
- build a prompt with the search results,
- ask the LLM to answer.

That means the develop

'The **agentic loop** is the repeated cycle where the LLM is allowed to decide what to do next:\n\n1. you send the user question plus history to the model,\n2. the model may return a **function call** instead of a final answer,\n3. your code runs that tool,\n4. you send the tool result back to the model,\n5. repeat until the model returns an answer with **no more tool calls**.\n\nSo the LLM is in the driver’s seat. The loop keeps going until it decides it has enough information. The course calls this the core of an agent: **instructions + tools + memory**, all wired together in a `while` loop.\n\n### How it differs from plain RAG\n\n**Plain RAG** is fixed:\n\n- search once,\n- build a prompt with the search results,\n- ask the LLM to answer.\n\nThat means the developer decides the flow up front. If the search misses because of a typo, awkward wording, or need for multiple searches, there’s **no recovery**.\n\n**Agentic RAG** is flexible:\n\n- the LLM can search,\n- inspect the results,

## Q6: How many times did the agent call search?
## Answer: 4